In [9]:
import os
import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import normalize
import google.generativeai as genai  # [변경] OpenAI -> Google Generative AI
from dotenv import load_dotenv

# [1] 환경 변수 로드 (.env 파일에서 키를 꺼내옴)
load_dotenv() 

# [변경] 시스템 환경 변수에서 Google API KEY를 가져옴
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# 키 확인용 안전장치
if not GOOGLE_API_KEY:
    raise ValueError("API Key를 찾을 수 없습니다. .env 파일에 'GOOGLE_API_KEY'가 있는지 확인해주세요.")

# [변경] Gemini 설정 및 모델명 변경
genai.configure(api_key=GOOGLE_API_KEY)
EMBEDDING_MODEL = "models/text-embedding-004" 

# [2] 사용자 제공 데이터셋 (TONE_DICT)
TONE_DICT = {
    'Scientific': ['연구', '데이터', '기술', '특허', '임상', '메커니즘', '효능', '성분', '솔루션', '혁신', '분석', '검증'],
    'Emotional': ['사랑', '행복', '마음', '위로', '함께', '추억', '소중한', '느낌', '감동', '선물', '일상', '여유'],
    'Luxury': ['프리미엄', '고품격', '가치', '특별한', '최고', '럭셔리', '노블레스', '장인', '헤리티지', '압도적'],
    'Casual': ['진짜', '대박', '완전', '가성비', '꿀팁', '그냥', '솔직', '친구', '쉬운', '간편', '추천']
}

# [3] [변경] Gemini 연동 임베딩 함수
def get_embedding_gemini(text_list):
    try:
        # Gemini는 리스트를 한 번에 받아 배치를 처리할 수 있습니다.
        # task_type="clustering"을 주면 군집화/유사도 분석에 최적화된 벡터를 줍니다.
        result = genai.embed_content(
            model=EMBEDDING_MODEL,
            content=text_list,
            task_type="clustering"
        )
        
        # 결과 딕셔너리에서 'embedding' 키에 벡터 리스트가 들어있습니다.
        return result['embedding']

    except Exception as e:
        print(f"❌ API 호출 중 오류 발생: {e}")
        # 오류 발생 시 빈 리스트 대신 에러를 raise하여 중단하거나, 
        # 필요시 0으로 채운 벡터를 반환하도록 예외처리를 추가할 수 있습니다.
        raise e

# [4] 벡터 생성 로직
def build_tone_assets(tone_dict):
    tone_vectors = {}
    meta_rows = []

    print(f"🚀 Start processing {len(tone_dict)} tones using Gemini API...")

    for tone_label, keywords in tone_dict.items():
        # [변경] 함수 이름 변경
        embeddings = get_embedding_gemini(keywords)
        embeddings = np.array(embeddings)
        
        # Centroid & Normalization
        centroid = np.mean(embeddings, axis=0)
        
        # 벡터 크기가 0인 경우(모든 키워드 임베딩 실패 등)에 대한 방어 코드
        if np.linalg.norm(centroid) > 0:
            centroid_norm = normalize(centroid.reshape(1, -1), norm='l2')[0]
        else:
            centroid_norm = centroid

        tone_vectors[tone_label] = centroid_norm
        
        meta_rows.append({
            "tone_id": tone_label,
            "keyword_count": len(keywords),
            "keywords": ", ".join(keywords),
            "model_used": EMBEDDING_MODEL
        })
        
        print(f" -> ✅ '{tone_label}' complete.")

    return tone_vectors, pd.DataFrame(meta_rows)

# [5] 실행
if __name__ == "__main__":
    try:
        vectors_pkl, df_meta = build_tone_assets(TONE_DICT)
        
        with open("tone_vectors.pkl", "wb") as f:
            pickle.dump(vectors_pkl, f)
        print("\n📂 [Saved] 'tone_vectors.pkl' saved successfully.")
        
        df_meta.to_csv("tone_metadata.csv", index=False, encoding="utf-8-sig")
        print("📂 [Saved] 'tone_metadata.csv' saved successfully.")
        
    except Exception as e:
        print(f"\n❌ 작업 중단됨: {e}")

🚀 Start processing 4 tones using Gemini API...
 -> ✅ 'Scientific' complete.
 -> ✅ 'Emotional' complete.
 -> ✅ 'Luxury' complete.
 -> ✅ 'Casual' complete.

📂 [Saved] 'tone_vectors.pkl' saved successfully.
📂 [Saved] 'tone_metadata.csv' saved successfully.
